In [ ]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader, TextLoader, UnstructuredFileLoader
import pandas as pd
from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json
from langchain.load import dumps, loads
from operator import itemgetter
from langchain.vectorstores import FAISS

In [ ]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'

### Remodel du fichier CSV Scraped Companies

In [ ]:
# # Chemin du fichier d'entrée
# input_file_path = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_M-A\\Data\\scraped_companies_all_columns.csv"

# # Chemin du fichier de sortie
# output_file_path = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_M-A\\Data\\scraped_companies_reorganized.csv"

# # Charger le fichier CSV
# df = pd.read_csv(input_file_path)

# # Renommer les colonnes
# df.columns = [
#     "Logo", "Column_to_remove", "Société", "Pays Siege",
#     "Siège répertorié sur Arx", "Site internet", "Secteur",
#     "Mots-clés", "Description", "Dernier CA (M)", "Périmètre CA",
#     "Column_to_remove_2", "TCAM (%)", "Dirigeants/Equipe CF", "Cotation", "ISIN"
# ]


# # Sélectionner et réorganiser les colonnes selon la spécification
# columns_to_keep = [
#     "Société",
#     "Pays Siege",
#     "Siège répertorié sur Arx",
#     "Site internet",
#     "Secteur",
#     "Mots-clés",
#     "Description",
#     "Dernier CA (M)",
#     "Périmètre CA",
#     "TCAM (%)",
#     "Dirigeants/Equipe CF",
#     "Cotation",
#     "ISIN"
# ]

# # Renommer les colonnes sélectionnées pour plus de clarté (facultatif)
# renamed_columns = [
#     "Company", "Country Headquarters", "Arx Listed HQ", "Website",
#     "Sector", "Keywords", "Description", "Latest Revenue (M)",
#     "Revenue Scope", "CAGR (%)", "Executives/Team CF", "Rating", "ISIN"
# ]

# # Réorganiser et renommer les colonnes
# df_reorganized = df[columns_to_keep]
# df_reorganized.columns = renamed_columns

# # Enregistrer le fichier CSV modifié
# df_reorganized.to_csv(output_file_path, index=False, encoding="utf-8")

# print(f"Le fichier CSV a été réorganisé et enregistré sous : {output_file_path}")

### Remodel fichier Base fonds PE et Dette D&A

In [ ]:
# Chemin vers votre fichier Excel
excel_file = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\20240924_Base_fonds PE & Dette.xlsx"

# Lire le fichier Excel en spécifiant que la troisième ligne (index 2) contient les noms des colonnes
# Remplacez 'sheet_name=1' par le nom de la feuille si nécessaire
df_excel = pd.read_excel(excel_file, sheet_name=1, header=2)

# Supprimer la première colonne
df_excel = df_excel.iloc[:, 1:]

# Vérifiez les premières lignes pour vous assurer que la première colonne a bien été supprimée
print("Aperçu des données importées après suppression de la première colonne :")
print(df_excel.head())

# Chemin où sauvegarder le CSV
csv_file_new_fonds = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\fichier_fonds_suppl.csv"

# Sauvegarder en CSV avec les noms de colonnes corrects
df_excel.to_csv(csv_file_new_fonds, index=False, encoding='utf-8')
print(f"Fichier CSV sauvegardé à : {csv_file_new_fonds}")

### Concaténation fichiers fonds Arx + D&A

In [ ]:
# Chemins vers les fichiers CSV
old_csv = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\pe_firms_cleaned.csv"
new_csv = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\fichier_fonds_suppl.csv"
combined_csv = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\pe_firms_combined.csv"

# Lire les fichiers CSV
df_old = pd.read_csv(old_csv)
df_new = pd.read_csv(new_csv)

# Vérifiez les colonnes de df_new et harmonisez-les avec df_old
print("Colonnes dans pe_firms_cleaned.csv :", df_old.columns.tolist())
print("Colonnes dans fichier_fonds_suppl.csv :", df_new.columns.tolist())

# Ajouter les nouvelles colonnes manquantes dans df_old avec des valeurs manquantes
for col in df_new.columns:
    if col not in df_old.columns:
        df_old[col] = pd.NA

# Concaténer les DataFrames
df_combined = pd.concat([df_old, df_new], ignore_index=True)

# Sauvegarder le fichier combiné
df_combined.to_csv(combined_csv, index=False, encoding='utf-8')

print(f"Fichier combiné sauvegardé à : {combined_csv}")


### Chargement des fichiers dans docs pour préparer l'embedding + Embedding (Attention a ne pas le lancer a chaque fois !!!)

Verifier quels fichiers mettre en commentaire et quel nom mettre en output pour ne rien écraser ni perdre

In [ ]:
import os
import json
import pandas as pd
from langchain.docstore.document import Document
from langchain.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
import openai


def load_files_with_metadata(file_paths, metadata_file):
    """
    Charge les fichiers et ajoute les métadonnées.
    Pour un CSV, chaque ligne devient un Document avec metadata (file_name, row_index, etc.)
    """
    with open(metadata_file, 'r', encoding="utf-8") as meta_file:
        metadata = json.load(meta_file)
    
    docs = []
    for path in file_paths:
        file_name = os.path.basename(path)
        file_metadata = metadata.get(file_name, {})
        
        if path.endswith(".csv"):
            df = pd.read_csv(path)
            # Chaque ligne devient un document
            for index, row in df.iterrows():
                text = "\n".join([f"{col}: {row[col]}" for col in df.columns])
                doc = Document(
                    page_content=text,
                    metadata={ "file_name": file_name, "row_index": index, **file_metadata }
                )
                docs.append(doc)
        elif path.endswith(".pdf"):
            loader = PyPDFLoader(path)
            for doc in loader.load():
                docs.append(Document(
                    page_content=doc.page_content,
                    metadata={ "file_name": file_name, **file_metadata }
                ))
        elif path.endswith(".txt"):
            # Spécifier l'encodage UTF-8 pour éviter l'erreur UnicodeDecodeError
            loader = TextLoader(path, encoding="utf-8")
            for doc in loader.load():
                docs.append(Document(
                    page_content=doc.page_content,
                    metadata={ "file_name": file_name, **file_metadata }
                ))
        else:
            # Gérer d'autres formats si nécessaire
            pass
    
    return docs
    

def generate_embeddings(file_paths, metadata_file, faiss_index_path):
    # Charger les documents avec métadonnées
    docs = load_files_with_metadata(file_paths, metadata_file)
    print(f"Nombre total de documents chargés : {len(docs)}")
    
    # Pour les CSV, les documents sont déjà assez courts.
    # Vous pouvez décider de ne pas splitter davantage ces documents.
    # Si vous souhaitez tout de même appliquer un text splitter pour les PDF ou TXT,
    # vous pouvez filtrer par type de document ou vérifier la longueur du contenu.
    final_docs = []
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
    for doc in docs:
        # Par exemple, ne splitter que si le document est trop long
        if len(doc.page_content) > 800:
            final_docs.extend(text_splitter.split_documents([doc]))
        else:
            final_docs.append(doc)
    
    print(f"Nombre de chunks après splitting : {len(final_docs)}")
    
    # Créer les embeddings et le vectorstore FAISS
    embedding = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(final_docs, embedding)
    
    # Sauvegarder l'index FAISS dans un dossier local
    vectorstore.save_local(faiss_index_path)
    print(f"Embeddings générés et stockés dans : {faiss_index_path}")

# Exemple d'utilisation pour une base de données complète :
all_files = [
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\pe_firms_combined_Arx.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\scraped_companies_reorganized_Arx.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\scraped_news_grid_Arx.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\exported_results.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_MergerMarket.csv",
    #"C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NLP.txt",
    "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NL_v2.txt"

]

metadata_file = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\metadata_arx.json"
faiss_index_path_all = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_2_400_0"

generate_embeddings(all_files, metadata_file, faiss_index_path_all)


### Embedding en faisant le split des fichiers trop lourds

In [9]:
import os
import json
import math
import shutil
import pandas as pd
from time import sleep
from tqdm import tqdm  # pip install tqdm
from langchain.docstore.document import Document
from langchain.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
import openai

# Configurez votre clé API OpenAI


def load_files_with_metadata(file_paths, metadata_file):
    """
    Charge les fichiers et ajoute les métadonnées.
    Pour un CSV, chaque ligne devient un Document avec metadata (file_name, row_index, etc.)
    """
    with open(metadata_file, 'r', encoding="utf-8") as meta_file:
        metadata = json.load(meta_file)
    
    docs = []
    for path in file_paths:
        file_name = os.path.basename(path)
        file_metadata = metadata.get(file_name, {})
        
        if path.endswith(".csv"):
            df = pd.read_csv(path)
            # Chaque ligne devient un document
            for index, row in df.iterrows():
                text = "\n".join([f"{col}: {row[col]}" for col in df.columns])
                doc = Document(
                    page_content=text,
                    metadata={"file_name": file_name, "row_index": index, **file_metadata}
                )
                docs.append(doc)
        elif path.endswith(".pdf"):
            loader = PyPDFLoader(path)
            for doc in loader.load():
                docs.append(Document(
                    page_content=doc.page_content,
                    metadata={"file_name": file_name, **file_metadata}
                ))
        elif path.endswith(".txt"):
            # Spécifier l'encodage UTF-8 pour éviter les problèmes d'encodage
            loader = TextLoader(path, encoding="utf-8")
            for doc in loader.load():
                docs.append(Document(
                    page_content=doc.page_content,
                    metadata={"file_name": file_name, **file_metadata}
                ))
        else:
            # Gérer d'autres formats si nécessaire
            pass
    
    return docs

def get_directory_size(directory):
    """Calcule la taille totale en octets d'un dossier et de ses sous-dossiers."""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(directory):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

def save_vectorstore_with_split(final_docs, embedding, base_faiss_index_path, max_size_mb=50, target_size_mb=49):
    """
    Crée un index FAISS à partir de final_docs.
    Si la taille totale de l'index (dossier) dépasse max_size_mb, 
    le corpus est divisé en plusieurs batches pour que chaque index soit inférieur à target_size_mb.
    """
    # Créer un index temporaire pour évaluer la taille
    temp_dir = base_faiss_index_path + "_temp"
    vectorstore_temp = FAISS.from_documents(final_docs, embedding)
    vectorstore_temp.save_local(temp_dir)
    
    total_size = get_directory_size(temp_dir)
    max_size_bytes = max_size_mb * 1024 * 1024
    target_size_bytes = target_size_mb * 1024 * 1024
    
    if total_size <= max_size_bytes:
        # Si la taille est acceptable, on déplace l'index temporaire vers la destination finale
        if os.path.exists(base_faiss_index_path):
            shutil.rmtree(base_faiss_index_path)
        os.rename(temp_dir, base_faiss_index_path)
        print(f"Index sauvegardé dans {base_faiss_index_path} avec une taille de {total_size/1024/1024:.2f} MB.")
    else:
        # Calculer le nombre de batches nécessaires
        num_batches = math.ceil(total_size / target_size_bytes)
        batch_size = math.ceil(len(final_docs) / num_batches)
        print(f"L'index est trop gros ({total_size/1024/1024:.2f} MB). "
              f"Division en {num_batches} batches d'environ {batch_size} documents chacun.")
        
        # Supprimer l'index temporaire
        shutil.rmtree(temp_dir)
        
        # Sauvegarder chaque batch dans un dossier séparé
        for i in range(num_batches):
            batch_docs = final_docs[i*batch_size:(i+1)*batch_size]
            batch_path = f"{base_faiss_index_path}_batch_{i+1}"
            vectorstore_batch = FAISS.from_documents(batch_docs, embedding)
            vectorstore_batch.save_local(batch_path)
            batch_size_bytes = get_directory_size(batch_path)
            print(f"Batch {i+1} sauvegardé dans {batch_path} avec {len(batch_docs)} documents, taille : {batch_size_bytes/1024/1024:.2f} MB.")

def generate_embeddings(file_paths, metadata_file, faiss_index_path):
    # Charger les documents avec métadonnées
    docs = load_files_with_metadata(file_paths, metadata_file)
    print(f"Nombre total de documents chargés : {len(docs)}")
    
    # Appliquer un text splitter pour uniformiser les chunks si nécessaire
    final_docs = []
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
    for doc in docs:
        if len(doc.page_content) > 800:
            final_docs.extend(text_splitter.split_documents([doc]))
        else:
            final_docs.append(doc)
    
    print(f"Nombre de chunks après splitting : {len(final_docs)}")
    
    # Instancier le modèle d'embedding
    embedding = OpenAIEmbeddings()
    
    # Créer et sauvegarder l'index en vérifiant la taille finale
    save_vectorstore_with_split(final_docs, embedding, faiss_index_path, max_size_mb=50, target_size_mb=49)

# --- Exemple d'utilisation ---
all_files = [
    # Listez ici les chemins de vos fichiers
    "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NLP.txt"
]
metadata_file = "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\metadata_arx.json"
faiss_index_path_all = "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0"

generate_embeddings(all_files, metadata_file, faiss_index_path_all)

Nombre total de documents chargés : 1
Nombre de chunks après splitting : 47857


C:\Users\namar.DA-CF\AppData\Local\Temp\ipykernel_10168\4293057816.py:128: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings()


L'index est trop gros (296.27 MB). Division en 7 batches d'environ 6837 documents chacun.
Batch 1 sauvegardé dans C:\Users\namar.DA-CF\OneDrive - D&A Corporate Finance\Documents\poc_RAG\Projet_test\RAG_MnA\Data\FAISS_index_actualites_NLP_400_0_batch_1 avec 6837 documents, taille : 42.34 MB.
Batch 2 sauvegardé dans C:\Users\namar.DA-CF\OneDrive - D&A Corporate Finance\Documents\poc_RAG\Projet_test\RAG_MnA\Data\FAISS_index_actualites_NLP_400_0_batch_2 avec 6837 documents, taille : 42.33 MB.
Batch 3 sauvegardé dans C:\Users\namar.DA-CF\OneDrive - D&A Corporate Finance\Documents\poc_RAG\Projet_test\RAG_MnA\Data\FAISS_index_actualites_NLP_400_0_batch_3 avec 6837 documents, taille : 42.35 MB.
Batch 4 sauvegardé dans C:\Users\namar.DA-CF\OneDrive - D&A Corporate Finance\Documents\poc_RAG\Projet_test\RAG_MnA\Data\FAISS_index_actualites_NLP_400_0_batch_4 avec 6837 documents, taille : 42.33 MB.
Batch 5 sauvegardé dans C:\Users\namar.DA-CF\OneDrive - D&A Corporate Finance\Documents\poc_RAG\Projet

### Retriever

Zone Test du retiever

In [ ]:
# Recharger le vectorstore depuis le répertoire persist_directory
faiss_index_path = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_2_400_0"
embedding = OpenAIEmbeddings()
vectorstore = FAISS.load_local(
        faiss_index_path, 
        embeddings=embedding, 
            allow_dangerous_deserialization=True
    )

print("VectorStore chargé depuis le disque.")

# Créer un système de récupération
# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={
#         "k": 10
#     }
# )


# # Créer un système de récupération
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Utiliser Maximal Marginal Relevance
    search_kwargs={
        "k": 5,  # Récupérer plus de documents
        "score_threshold": 0.01  # Réduire le seuil de score pour inclure plus de résultats
    }
)

# Exemple de requête
query = "Dans quels deals a été impliquée BNP Paribas"
results = retriever.get_relevant_documents(query)

print("\nRésultats de la requête :")
for result in results:
    print(result.page_content)



### Génération avec LLM

### RAG MULTI QUERY

Generation de plusieurs reformulations de la requete initiale pour avoir plus d'occurences dans les docs retrievés

In [ ]:
# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate ten 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)


generate_queries = (
    prompt_perspectives 
    | ChatOpenAI(model="gpt-5-mini") 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]
 
# Retrieve
question = "Quelles nouveautées pour Vulcain ingénierie ?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})

# RAG
template = """Tu es un assistant chatbot qui travaille dans un cabinet de finance d'entreprise. Ton rôle est de donner les informations les plus pertinentes possibles en te basant sur les sources que tu as. Voici le contexte pour t'aider:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-5-mini")

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

### RAG FUSION

Rank les documents trouvés par le multi query pour avoir une hierarchie

In [ ]:
# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

question = "Fais moi une fiche société sur Cardinet ?"

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(model="gpt-5-mini")
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
llm = ChatOpenAI(model="gpt-5-mini")

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

### RAG FUSION ACTUALITES

Application directe du rag fusion pour la base de données actualités de CFNews et Arx

In [ ]:
def rag_fusion_actualites(question: str) -> str:
    # Configurer les clés API
    # Charger les clés API depuis les variables d'environnement
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_API_KEY'] = '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'

    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

    faiss_index_path = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites"
    embedding = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(
            faiss_index_path, 
            embeddings=embedding, 
            allow_dangerous_deserialization=True
        )

    # Créer un système de récupération
    # retriever = vectorstore.as_retriever(
    #     search_type="mmr",
    #     search_kwargs={
    #         "k": 10
    #     }
    # )
    
    retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.6, "k":15}
    )

    # Génération des requêtes
    query_generation_template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
    Generate multiple search queries related to: {question} \n
    Output (4 queries):"""
    prompt_rag_fusion = ChatPromptTemplate.from_template(query_generation_template)

    generate_queries = (
        prompt_rag_fusion
        | ChatOpenAI(model="gpt-5-mini")
        | StrOutputParser()
        | (lambda x: x.split("\n"))  # Liste des requêtes
    )

    # Étape 1 : Générer les requêtes
    queries = generate_queries.invoke({"question": question})
    print(f"Requêtes générées : {queries}")

    # Étape 2 : Récupération des documents
    results = [retriever.invoke(q) for q in queries]
    print(f"Documents récupérés : {results}")

    # Étape 3 : Fusion Reciprocal Rank Fusion
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            # Convertir le Document en dict avant la sérialisation
            doc_dict = {
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }
            doc_str = dumps(doc_dict)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + 100)

    reranked_docs = [
        (Document(page_content=d["page_content"], metadata=d["metadata"]), score)
        for d_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        for d in [loads(d_str)]
        ]    
    
    print(f"Documents fusionnés : {len(reranked_docs)} documents rerankés.")

    # Préparer le contexte pour le modèle LLM
    context = "\n\n".join([doc.page_content for doc, _ in reranked_docs])

    # Étape 4 : Répondre à la question
    answer_template = """Answer the following question based on this context :

    {context}

    Question: {question}
    """
    answer_prompt = ChatPromptTemplate.from_template(answer_template)
    llm = ChatOpenAI(model="gpt-5-mini")

    # Génération de la réponse
    final_input = {"context": context, "question": question}
    answer = (
        answer_prompt
        | llm
        | StrOutputParser()
    ).invoke(final_input)

    return answer

print(rag_fusion_actualites(question="Donne moi tous les deals dans lesquels est impliqué Vulcain"))

AttributeError: module 'openai' has no attribute 'OpenAI'

Si la base de données est trop grosse (et pour éventuellement tester la variante que de tester sur une grosse bdd), split des embeddings en plusieurs bases de données et rag fusion sur chacune.

In [ ]:
import os
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain.chat_models import ChatOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langchain.load import dumps, loads

# --- Monkey-patch (si vous ne pouvez pas upgrader openai) ---
if not hasattr(openai, "OpenAI"):
    openai.OpenAI = openai.Client
if not hasattr(openai, "AsyncOpenAI") and hasattr(openai, "AsyncClient"):
    openai.AsyncOpenAI = openai.AsyncClient

def rag_fusion_actualites(question: str) -> str:
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

    # 1) Batches FAISS
    batch_paths = [
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_1",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_2",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_3",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_4",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_5",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_6",
        r"C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_7",
    ]

    # 2) Embeddings & retrievers
    embedding = OpenAIEmbeddings()
    retrievers = []
    for path in batch_paths:
        vs = FAISS.load_local(
            path,
            embeddings=embedding,
            allow_dangerous_deserialization=True
        )
        retrievers.append(
            vs.as_retriever(
                search_type="similarity_score_threshold",
                search_kwargs={"score_threshold": 0.7, "k": 10}
            )
        )

    # 3) Générer 4 requêtes (temperature=1 pour gpt-5-mini)
    query_tpl = ChatPromptTemplate.from_template(
        "You are a helpful assistant that generates 4 search queries based "
        "on the input. Generate 4 queries related to: {question}"
    )
    raw = (
        query_tpl
        | ChatOpenAI(model="gpt-5-mini", temperature=1)
        | StrOutputParser()
    ).invoke({"question": question})
    queries = [q.strip() for q in raw.split("\n") if q.strip()]

    # 4) Parallel retrieval
    def retrieve(r, q):
        return r.invoke(q)

    all_results = []
    for q in queries:
        with ThreadPoolExecutor(max_workers=len(retrievers)) as exe:
            futures = [exe.submit(retrieve, r, q) for r in retrievers]
            docs = []
            for fut in as_completed(futures):
                docs.extend(fut.result())
        all_results.append(docs)

    # 5) RRF fusion
    fused = {}
    for docs in all_results:
        for rank, doc in enumerate(docs):
            key = dumps({"page_content": doc.page_content, "metadata": doc.metadata})
            fused.setdefault(key, 0.0)
            fused[key] += 1.0 / (rank + 100)

    reranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    reranked_docs = [
        (Document(**loads(doc_str)), score)
        for doc_str, score in reranked
    ]

    # 6) Contexte & réponse finale (temperature=1)
    context = "\n\n".join(d.page_content for d, _ in reranked_docs)
    answer_tpl = ChatPromptTemplate.from_template(
        "Answer the question based on this context, then complete with your knowledge:\n\n"
        "{context}\n\nQuestion: {question}"
    )
    answer = (
        answer_tpl
        | ChatOpenAI(model="gpt-5-mini", temperature=1)
        | StrOutputParser()
    ).invoke({
        "context": context,
        "question": question
    })
    return answer

# Exécution
print(rag_fusion_actualites("Donne moi tous les deals dans lesquels est impliqué Vulcain"))



C:\Users\namar.DA-CF\AppData\Local\Temp\ipykernel_23192\2271665075.py:84: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (Document(**loads(doc_str)), score)


D'après le contexte fourni, voici l'ensemble des transactions impliquant **Vulcain** :

1. **4 juillet 2022** :
   - **Acheteur** : SPIRAX SARCO
   - **Vendeur** : QUALIUM INVESTISSEMENT et le fonds FUTURE FRENCH CHAMPIONS
   - **Transaction** : Acquisition de **VULCANIC**, une entreprise spécialisée dans les solutions de chauffage et de refroidissement pour l'industrie, située en Île-de-France.
   - **Chiffre d'affaires de VULCANIC en 2021** : 89,40 millions d'euros
   - **Valorisation de la transaction** : 262 millions d'euros

2. **31 juillet 2023** :
   - **Acheteur** : **VULCAIN ENGINEERING**
   - **Vendeur** : IPLAN GESTION INTEGRAL
   - **Transaction** : Acquisition d'IPLAN GESTION INTEGRAL, une entreprise espagnole spécialisée dans les services d’ingénierie dans le secteur de l'énergie et des travaux publics.
   - **Valorisation de l'opération** : Entre 0 et 20 millions d'euros

3. **10 juillet 2023** :
   - **Acheteur** : **VULCAIN ENGINEERING**
   - **Vendeur** : APSALYS (fon

### RAG FUSION FONDS

Rag fusion appliqué à la base de données fonds d'investissements (concaténation de D&A et Arx)

In [ ]:
def rag_fusion_fonds(question: str) -> str:
    # Configurer les clés API
    # Charger les clés API depuis les variables d'environnement
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_API_KEY'] = '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'

    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

    faiss_index_path = "C:\\Users\\namar\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_fonds"
    embedding = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(
            faiss_index_path, 
            embeddings=embedding, 
            allow_dangerous_deserialization=True
        )

    # Créer un système de récupération
    retriever = vectorstore.as_retriever(
        search_type="mmr",  # Utiliser Maximal Marginal Relevance
        search_kwargs={
            "k": 10,  # Récupérer plus de documents
            "score_threshold": 0.01  # Réduire le seuil de score pour inclure plus de résultats
        }
    )

    # Génération des requêtes
    query_generation_template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
    Generate multiple search queries related to: {question} \n
    Output (4 queries):"""
    prompt_rag_fusion = ChatPromptTemplate.from_template(query_generation_template)

    generate_queries = (
        prompt_rag_fusion
        | ChatOpenAI(model="gpt-5-mini")
        | StrOutputParser()
        | (lambda x: x.split("\n"))  # Liste des requêtes
    )

    # Étape 1 : Générer les requêtes
    queries = generate_queries.invoke({"question": question})
    print(f"Requêtes générées : {queries}")

    # Étape 2 : Récupération des documents
    results = [retriever.invoke(q) for q in queries]
    print(f"Documents récupérés : {results}")

    # Étape 3 : Fusion Reciprocal Rank Fusion
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            # Convertir le Document en dict avant la sérialisation
            doc_dict = {
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }
            doc_str = dumps(doc_dict)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + 60)

    reranked_docs = [
        (Document(page_content=d["page_content"], metadata=d["metadata"]), score)
        for d_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        for d in [loads(d_str)]
        ]    
    
    print(f"Documents fusionnés : {len(reranked_docs)} documents rerankés.")

    # Préparer le contexte pour le modèle LLM
    context = "\n\n".join([doc.page_content for doc, _ in reranked_docs])

    # Étape 4 : Répondre à la question
    answer_template = """Answer the following question based on this context:

    {context}

    Question: {question}
    """
    answer_prompt = ChatPromptTemplate.from_template(answer_template)
    llm = ChatOpenAI(model="gpt-5-mini")

    # Génération de la réponse
    final_input = {"context": context, "question": question}
    answer = (
        answer_prompt
        | llm
        | StrOutputParser()
    ).invoke(final_input)

    return answer

In [ ]:
import os
import json
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
from langchain_core.output_parsers import StrOutputParser
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def rag_fusion_fiche_societe_to_word_websearch(question: str) -> dict:
    print("[LOG] Démarrage pour :", question)

    # 1) Vos 7 batches FAISS
    batch_dirs = [
        "./Data/FAISS_index_actualites_NLP_400_0_batch_1",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_2",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_3",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_4",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_5",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_6",
        "./Data/FAISS_index_actualites_NLP_400_0_batch_7",
    ]

    # 2) Chargez tous les FAISS en parallèle
    embedding = OpenAIEmbeddings()
    retrievers = []
    with ThreadPoolExecutor(max_workers=len(batch_dirs)) as exe:
        futures = {
            exe.submit(
                FAISS.load_local,
                path,
                embeddings=embedding,
                allow_dangerous_deserialization=True
            ): path
            for path in batch_dirs
        }
        for fut in as_completed(futures):
            path = futures[fut]
            try:
                vs = fut.result()
                retrievers.append(
                    vs.as_retriever(
                        search_type="similarity_score_threshold",
                        search_kwargs={"score_threshold": 0.6, "k": 10}
                    )
                )
                print(f"[LOG] Chargé FAISS depuis {path}")
            except Exception as e:
                print(f"[ERROR] échec chargement {path} : {e}")
    print(f"[LOG] {len(retrievers)} retrievers prêts.")

    # 3) Génération de 3 requêtes
    prompt_q = f"""
You are a helpful assistant that generates 3 distinct search queries based on the input.
Input: {question}

Output the 3 queries, one per line:
""".strip()
    resp_q = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt_q}],
    )
    raw_q = resp_q.choices[0].message.content
    queries = [q.strip() for q in raw_q.splitlines() if q.strip()]
    print("[LOG] Queries générées :", queries)

    # 4) Récupération parallèle + Fusion RRF par requête
    all_docs = []
    for q in queries:
        with ThreadPoolExecutor(max_workers=len(retrievers)) as exe:
            futs = [exe.submit(r.invoke, q) for r in retrievers]
            docs = []
            for f in as_completed(futs):
                try:
                    docs.extend(f.result())
                except Exception as e:
                    print(f"[ERROR] retrieval '{q}' failed : {e}")
        all_docs.append(docs)

    # 5) Reciprocal Rank Fusion global
    fused = {}
    for docs in all_docs:
        for rank, doc in enumerate(docs, start=1):
            key = json.dumps({"page_content": doc.page_content, "metadata": doc.metadata})
            fused[key] = fused.get(key, 0) + 1.0 / (rank + 100)
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    reranked_docs = [Document(**json.loads(k)) for k, _ in ranked]
    print(f"[LOG] RRF fusion aboutit à {len(reranked_docs)} docs.")

    # 6) Contexte abrégé (10 premiers)
    context = "\n\n".join(doc.page_content for doc in reranked_docs[:10])

    # 7) Prompt final (ici on échappe le JSON schema avec doubles {{ }})
    answer_template = r"""
You are a financial journalist and M&A expert. You MUST answer in JSON format only, strictly matching the provided structure.

Use primarily the context provided below (abridged) to construct your answer. If the context lacks certain details, supplement your answer with the most relevant results from your internet search.
Provide the URL of the source when you give a number!
Context (abridged):
{context}

Question: {question}

Respond ONLY within the following JSON structure (no extra text):

{{
    "nom_societe": "Provide the company name if found",
    "description_activite": "Provide a detailed description of the company's activities (5-10 lines), using clear and concise language.",
    "chiffres_cles": "Include key metrics such as revenue, employee count, or founding date, summarized if needed.",
    "clients_par_secteur": "List the main clients by sector and give their names.",
    "implantation_positionnement": "List cities or countries where the company is located.",
    "elements_financiers": "Summarize the financial growth over the past 3 years in a concise manner.",
    "president": "Name of the president",
    "daf": "Name of the financial director",
    "actionnaire": "Provide a summarized list of key shareholders or investment funds, with the most important ones highlighted.",
    "actionnaire_pourcentage": "Shareholder distribution percentages if available.",
    "creanciers_type": "List types of creditors concisely.",
    "creanciers_commentaires": "Provide a brief summary of the creditors' comments.",
    "actualites_presse": "Present recent press news with maximum details and clear language.",
    "equity_story": "Present major equity events and investments (e.g., LBO, MBO) with maximum details and clear language.",
    "creation": "Present the company's creation details or founding year with maximum details and clear language.",
    "acquisitions": "Present all key acquisitions, build-ups, mergers with dates and descriptions with maximum details and clear language."
}}
""".strip()

    prompt_final = answer_template.format(context=context, question=question)
    # 8) Appel search-preview **sans** temperature
    resp_f = openai.chat.completions.create(
        model="gpt-4o-mini-search-preview",
        messages=[{"role": "user", "content": prompt_final}],
    )
    raw_f = resp_f.choices[0].message.content

    # 9) Nettoyage + JSON parse
    text = raw_f.strip()
    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print("[ERROR] impossible de parser JSON :", e)
        return {}
    
# Exemple d’usage :
if __name__ == "__main__":
    fiche = rag_fusion_fiche_societe_to_word_websearch(
        "Fournis-moi une fiche détaillée pour l'entreprise Setic Pourtier."
    )
    print(json.dumps(fiche, indent=2, ensure_ascii=False))


[LOG] Démarrage pour : Fournis-moi une fiche détaillée pour l'entreprise Setic Pourtier.
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_2
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_5
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_7
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_3
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_4
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_1
[LOG] Chargé FAISS depuis ./Data/FAISS_index_actualites_NLP_400_0_batch_6
[LOG] 7 retrievers prêts.
[LOG] Queries générées : ["1. Fiche détaillée de l'entreprise Setic Pourtier", '2. Informations complètes sur Setic Pourtier', '3. Profil entreprise Setic Pourtier']
[LOG] RRF fusion aboutit à 153 docs.
{
  "nom_societe": "Setic Pourtier",
  "description_activite": "Setic Pourtier est une entreprise française spécialisée dans la conception et la fabrication de mach

### Conversion données tabulaires Actualités en texte

Passage en phrase du fichier tabulaire actualités CFNews pour que l'embedding soit de meilleure qualité ? Perf un peu meilleures mais ca reste pas fou.

Piste : tester des modèles d'embedding adaptés aux bases de données sous forme tabulaire.

In [ ]:
import os
import openai
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# 1) Chemins (double backslashes)
CSV_FILE = (
    "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\"
    "Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews.csv"
)
OUTPUT_TXT = (
    "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\"
    "Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NLP.txt"
)

# 2) API Key
openai.api_key = os.getenv("OPENAI_API_KEY")

def get_processed_lines(txt_path: str) -> set[int]:
    seen = set()
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("Ligne "):
                    try:
                        n = int(line.split(":",1)[0].replace("Ligne","").strip())
                        seen.add(n)
                    except ValueError:
                        continue
    return seen

def generate_from_text(row_number: int, text: str) -> tuple[int, str]:
    prompt = (
        "Transforme le paragraphe suivant en phrases complètes et cohérentes qui "
        "décrivent entièrement l'information. Détaille autant que tu peux, en "
        "rappelant au moins 2 fois le nom de l'acheteur et du vendeur.\n\n"
        f"{text}\n\n"
    )
    try:
        resp = openai.ChatCompletion.create(
            model="gpt-5-mini",
            messages=[{"role":"user","content":prompt}],
            temperature=0.7,
            max_tokens=400
        )
        return row_number, resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"❗ Erreur génération L{row_number}: {e}")
        return row_number, ""

def main():
    # 3) Lecture du CSV « une seule colonne »
    df = pd.read_csv(
        CSV_FILE,
        sep=",",
        header=None,
        names=["Formatted_Row"],
        dtype=str,
        engine="python",
        on_bad_lines="skip"
    )
    total = len(df)

    # 4) Ligne de départ
    processed = get_processed_lines(OUTPUT_TXT)
    start_idx = max(processed) if processed else 0
    print(f"--> Démarrage à la ligne {start_idx+1}/{total}")

    # 5) Préparation des tâches
    tasks = [
        (idx+1, df.iloc[idx]["Formatted_Row"])
        for idx in range(start_idx, total)
    ]

    new_entries = []
    # 6) Parallelisation avec 5 threads
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {
            executor.submit(generate_from_text, row_num, txt): row_num
            for row_num, txt in tasks
        }
        for future in tqdm(as_completed(futures),
                           total=len(futures),
                           desc="NLP ThreadPool"):
            row_num, sentence = future.result()
            if sentence:
                print(f"\n--- Ligne {row_num} → {sentence}\n")
                new_entries.append((row_num, sentence))

    # 7) Sauvegarde
    if new_entries:
        with open(OUTPUT_TXT, "a", encoding="utf-8") as f:
            for num, txt in sorted(new_entries):
                f.write(f"Ligne {num}:\n{txt}\n\n")

    print(f"✅ {len(new_entries)} lignes NLP ajoutées.")
    print(f"Fichier : {OUTPUT_TXT}")

if __name__ == "__main__":
    main()


In [ ]:
import re
import json
import pandas as pd
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever  # Vérifiez que votre version de LangChain fournit cette classe
from langchain_openai import OpenAIEmbeddings

# Fonction pour charger les documents NLP depuis un fichier texte
def load_nlp_text_documents(file_path: str) -> list:
    """
    Lit un fichier texte où chaque section commence par "Ligne <num>:" et renvoie une
    liste de Documents. Chaque Document contiendra la description narrative d'une transaction.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    # Découper le fichier à chaque occurrence de "Ligne <num>:"
    segments = re.split(r'\n\s*Ligne \d+:', content)
    documents = []
    for segment in segments:
        seg = segment.strip()
        if seg:
            documents.append(Document(page_content=seg, metadata={}))
    return documents

# Exemple de création de la variable `documents` depuis votre fichier
file_path = "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NLP.txt"
documents = load_nlp_text_documents(file_path)

# Définir l'embedding (assurez-vous que votre variable embedding est définie avant usage)
embedding = OpenAIEmbeddings()

# Charger un index FAISS existant (assurez-vous que le chemin est correct)
faiss_index = FAISS.load_local(
    "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_3",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)

bm25_retriever = BM25Retriever.from_documents(documents, k=10)

def hybrid_retrieval(query: str, faiss_index: FAISS, bm25_retriever: BM25Retriever, alpha: float = 0.5) -> list:
    """
    Effectue une recherche hybride (dense via FAISS et sparse via BM25) puis fusionne les résultats.
    Le paramètre alpha (entre 0 et 1) définit la pondération de la recherche dense.
    Retourne une liste de Documents classés par pertinence.
    """
    # Recherche dense avec FAISS
    dense_results = faiss_index.similarity_search(query, k=50)
    
    # Recherche sparse avec BM25
    sparse_results = bm25_retriever.invoke(query)  # Ici, on suppose que cette méthode renvoie une liste de Documents
    
    # Fusionner les scores avec une approche de Reciprocal Rank Fusion (RRF)
    fused_scores = {}
    
    for rank, doc in enumerate(dense_results):
        doc_id = doc.page_content  # On utilise le contenu comme identifiant (dans une application, il est préférable d'utiliser un ID unique)
        if doc_id not in fused_scores:
            fused_scores[doc_id] = 0
        fused_scores[doc_id] += alpha / (rank + 1)
    
    for rank, doc in enumerate(sparse_results):
        doc_id = doc.page_content
        if doc_id not in fused_scores:
            fused_scores[doc_id] = 0
        fused_scores[doc_id] += (1 - alpha) / (rank + 1)
    
    # Éliminer les doublons et reconstruire la liste de Documents
    unique_docs = {doc.page_content: doc for doc in (dense_results + sparse_results)}
    ranked_docs = sorted(unique_docs.values(), key=lambda d: fused_scores.get(d.page_content, 0), reverse=True)
    
    return ranked_docs

# Exemple d'utilisation de la recherche hybride
ranked_results = hybrid_retrieval("Donne moi les deals de Inetum", faiss_index, bm25_retriever, alpha=0.7)

for doc in ranked_results:
    print(doc.page_content)


In [ ]:
import re
import json
import pandas as pd
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever  # Assurez-vous que cette classe est disponible
from langchain_openai import OpenAIEmbeddings

def load_nlp_text_documents(file_path: str) -> list:
    """
    Lit un fichier texte où chaque section commence par "Ligne <num>:" et renvoie une
    liste de Documents.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    segments = re.split(r'\n\s*Ligne \d+:', content)
    documents = []
    for segment in segments:
        seg = segment.strip()
        if seg:
            documents.append(Document(page_content=seg, metadata={}))
    return documents

def count_occurrences(documents: list, keyword: str) -> int:
    """
    Incrémente le compteur d'un document si le mot-clé apparaît au moins une fois dans le contenu.
    Le comptage se fait de manière insensible à la casse.
    """
    count = 0
    for doc in documents:
        if keyword.lower() in doc.page_content.lower():
            count += 1
    return count

def print_documents_with_keyword(documents: list, keyword: str):
    """
    Affiche les documents qui contiennent le mot-clé, en montrant les 300 premiers caractères de leur contenu.
    """
    matching_docs = [doc for doc in documents if keyword.lower() in doc.page_content.lower()]
    print(f"\nNombre de documents contenant '{keyword}': {len(matching_docs)}")
    for i, doc in enumerate(matching_docs, start=1):
        print(f"\n--- Document {i} ---")
        content = doc.page_content
        if len(content) > 300:
            print(content[:300] + "...")
        else:
            print(content)

def hybrid_retrieval(query: str, faiss_index: FAISS, bm25_retriever: BM25Retriever, alpha: float = 0.5) -> list:
    """
    Effectue une recherche hybride (dense via FAISS et sparse via BM25) puis fusionne les résultats.
    
    Le paramètre alpha (entre 0 et 1) définit la pondération de la recherche dense,
    et (1 - alpha) pour la recherche sparse.
    
    Retourne une liste de Documents classés par pertinence.
    """
    # Recherche dense (vecteurs)
    dense_results = faiss_index.similarity_search(query, k=50)
    
    # Recherche sparse (BM25)
    sparse_results = bm25_retriever.invoke(query)  # Suppose que cette méthode renvoie une liste de Documents
    
    # Fusionner les scores avec Reciprocal Rank Fusion (RRF)
    fused_scores = {}
    for rank, doc in enumerate(dense_results):
        doc_id = doc.page_content  # On utilise le contenu comme identifiant
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + alpha / (rank + 1)
    for rank, doc in enumerate(sparse_results):
        doc_id = doc.page_content
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + (1 - alpha) / (rank + 1)
    
    # Eliminer les doublons
    unique_docs = {doc.page_content: doc for doc in (dense_results + sparse_results)}
    ranked_docs = sorted(unique_docs.values(), key=lambda d: fused_scores.get(d.page_content, 0), reverse=True)
    
    return ranked_docs

# Exemple d'utilisation :

# Charger les documents NLP à partir du fichier (mettre ici le chemin avec des doubles backslashes)
file_path = "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\deals_data_cleaned_CFNews_converted_NLP.txt"

# Charger les documents NLP
documents = load_nlp_text_documents(file_path)

# Initialiser l'embedding
embedding = OpenAIEmbeddings()

# Charger l'index FAISS (dense)
faiss_index = FAISS.load_local(
    "C:\\Users\\namar.DA-CF\\OneDrive - D&A Corporate Finance\\Documents\\poc_RAG\\Projet_test\\RAG_MnA\\Data\\FAISS_index_actualites_NLP_400_0_batch_3",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)

# Créer le BM25Retriever en passant la liste de documents via 'docs'
bm25_retriever = BM25Retriever.from_documents(documents, k=10)

# Recherche pure dense
dense_results = faiss_index.similarity_search("Donne moi tous les deals dans lesquels est impliqué Vulcain", k=50)
dense_count = count_occurrences(dense_results, "Vulcain")
print("Nombre de documents avec 'BNP' (dense uniquement):", dense_count)
print_documents_with_keyword(dense_results, "Vulcain")

# Recherche hybride (dense + BM25)
hybrid_results = hybrid_retrieval("Donne moi tous les deals dans lesquels est impliqué Vulcain", faiss_index, bm25_retriever, alpha=0.7)
hybrid_count = count_occurrences(hybrid_results, "Vulcain")
print("\nNombre de documents avec 'BNP' (hybride):", hybrid_count)
print_documents_with_keyword(hybrid_results, "Vulcain")
